# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object - access via attributes
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll list all available record sets in the dataset, then inspect the fields and columns within one.

In [ ]:
# List all record sets present in the metadata
# Each record set's '@id' uniquely identifies it
record_sets = metadata.record_sets
print('Available Record Sets:')
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '<no name>')}")

# If no record sets found, probe file objects for possible data tables
if not record_sets:
    print('\nNo record sets found in the metadata. Attempting to list file objects...')
    if hasattr(metadata, 'file_objects'):
        for fo in metadata.file_objects:
            print(f"  - FileObject @id: {fo['@id']}, name: {fo.get('name', '<no name>')}, encodingFormat: {fo.get('encodingFormat', '<none>')}")
    else:
        print('No file objects available either.')

Let's inspect the fields of one of the record sets (or file objects if record sets not present) by their `@id` and name.

**Note:** All references to record sets, fields and columns use their `@id` for reproducible and unambiguous workflow.

In [ ]:
# For demonstration, select the first record set or file object if available
if record_sets:
    main_rs = record_sets[0]
    main_rs_id = main_rs['@id']
    print(f"Selected record set @id: {main_rs_id}\n")
    print("Fields in this record set:")
    for field in main_rs.get('fields', []):
        print(f"  - Field @id: {field['@id']}, name: {field.get('name','<no name>')}, dataType: {field.get('dataType','<none>')}")
else:
    file_objects = getattr(metadata, 'file_objects', [])
    if file_objects:
        main_file = file_objects[0]
        main_rs_id = main_file['@id']
        print(f"Selected file object @id: {main_rs_id}\n")
        print("Columns in this file:")
        for col in main_file.get('columns', []):
            print(f"  - Column @id: {col['@id']}, name: {col.get('name','<no name>')}, dataType: {col.get('dataType','<none>')}")
    else:
        main_rs_id = None
        print('No record sets or file objects found.')

## 3. Data Extraction
Load data from a specific record set or file object into a DataFrame for analysis.
Use the record set or file object and field/column `@id`s from the overview.

In [ ]:
# List all available record set/file object ids
record_set_ids = [rs['@id'] for rs in record_sets] if record_sets else [fo['@id'] for fo in getattr(metadata,'file_objects',[])]
dataframes = {}

# Extract and load records from each record set
for recset_id in record_set_ids:
    try:
        # The records() API expects @id of the record set or file object
        records = list(dataset.records(record_set=recset_id))
        dataframes[recset_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records from {recset_id}.")
    except Exception as e:
        print(f"Failed to load records from {recset_id}: {e}")

# Preview columns for the primary record set/file object
if main_rs_id in dataframes:
    print('Columns available:', dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section may include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

We'll demonstrate filtering, normalization, and grouping using columns identified by their `@id` (or, if the mapping is clear, by their name).

In [ ]:
# Use a numeric field for EDA. Adjust the field name based on the actual data columns loaded previously.
if main_rs_id in dataframes and not dataframes[main_rs_id].empty:
    df = dataframes[main_rs_id]
    # Try to find a likely numeric column for demonstration
    numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64'] or pd.api.types.is_numeric_dtype(df[col])]
    # If not found, try some common numeric field names
    trial_names = ['MW','Molecular Weight','Peptides','Coverage %','Abundance']
    chosen = None
    for name in trial_names:
        if name in df.columns:
            chosen = name
            break
    if not chosen and numeric_candidates:
        chosen = numeric_candidates[0]
    
    if chosen:
        numeric_field = chosen  # e.g., 'MW' or '@id:...MW'
        print(f"Using field: {numeric_field} for numeric EDA.")
        threshold = df[numeric_field].quantile(0.75)  # e.g., top 25% values
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())
    
        # Normalize
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())
    
        # Group by a potentially interesting field (e.g., 'Description', 'Protein', etc.)
        group_field = None
        for name in ['Description','Protein','Accession','Sample', 'Category']:
            if name in df.columns:
                group_field = name
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print('No suitable categorical field found for grouping.')
    else:
        print('No numeric column found to demonstrate EDA.')
else:
    print('Primary data frame is empty or missing.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the selected numeric field and a bar chart of group means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id in dataframes and not dataframes[main_rs_id].empty:
    df = dataframes[main_rs_id]
    if 'numeric_field' in locals() and numeric_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()
    
    if 'grouped_df' in locals() and group_field:
        top_groups = grouped_df.sort_values(numeric_field, ascending=False).head(10)
        plt.figure(figsize=(10,5))
        sns.barplot(data=top_groups, x=group_field, y=numeric_field)
        plt.xticks(rotation=60)
        plt.title(f"Top 10 {group_field} by average {numeric_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded the Croissant metadata and inspected the data structure by record set and fields (referenced using `@id`).
- Loaded record set data into DataFrames for processing and visualizations.
- Performed basic filtering, normalization, and grouping operations on protein mass spectrometry data.
- Data visualizations reveal statistical properties of numeric attributes (e.g., Molecular Weight distribution), with examples of aggregating by group.

Further work could include advanced protein annotation analysis, cross-referencing with external ontologies, and machine learning preparation for biological interpretation.